# PSELDNets — outdoor_siren_v6 学習（train 240 / val 80、クラス毎データ4倍増量版）

**v6 = v5と完全同一の構成・レンジで、クラス毎のデータ量だけ4倍にした版**:
4クラス（Siren / Horn / BackupBeep / BikeBell）× 各 train60 / val20 = 計320本。
速度・距離・SNR・SIR・発音区間レンジ、クリーン合成の妨害音、クラス辞書はすべてv5と同一。

**目的（v5 run2の誤り解剖の続き）**: v5では取り違え(substitution)ゼロ・missも僅少だった一方、
**方向誤差(dir_err>20°)が時間的に疎なクラスに集中**した（BackupBeep 44% / BikeBell 32% vs
Horn 0.6%）。機構は「発音の合間もSED検出は続くがDOAが漂う」。v5はクラス毎の学習データが
v3の1/4（train15本）だったため、この問題が
- **データ量で消える** → 学習不足が原因（増量で解決できる）
- **データ量でも残る** → 疎発音×移動音源の本質的限界（卒論の考察の柱として確定）
のどちらかを切り分けるのがv6の役割。

## ⚠️ 使用前に必ず確認

1. **ランタイム → ランタイムのタイプを変更 → T4 GPU** を選択
2. **Drive の `MyDrive/PSELDNets_data/` に `dataset_outdoor_siren_v6.zip` をアップロード済み**であること
3. セルを**上から順に**実行（途中でスキップしない）
4. 中断しても再実行すれば続きから学習が再開される（Drive永続化＋自動resume）
5. **注意**: データを差し替えて再学習するときは、必ず下の`EXP_NAME`を新しい名前に変える
   （固定experiment_nameのまま再実行すると、古いckptに新データを食わせて破綻する事故が
   v3のrun1で発生済み。詳細はPROGRESS.md「Colab run1のNaN崩壊とrun2再学習」節）

---
## 1. GPU 確認

In [ ]:
import torch
assert torch.cuda.is_available(), '⚠️ GPU未接続。ランタイムのタイプを T4 GPU に変更してください。'
print(f'PyTorch : {torch.__version__}')
print(f'GPU     : {torch.cuda.get_device_name(0)}')
print(f'VRAM    : {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB')

## 2. Drive マウントとパス設定

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

# ==== 設定（必要ならここだけ書き換える） ====
DRIVE_DATA = '/content/drive/MyDrive/PSELDNets_data'   # zip の置き場所
DRIVE_LOGS = '/content/drive/MyDrive/PSELDNets_logs'   # 学習ログ・ckpt の永続化先
DRIVE_CKPT = '/content/drive/MyDrive/PSELDNets_ckpts'  # 事前学習ckptのキャッシュ
DATASET    = 'outdoor_siren_v6'
EXP_NAME   = 'outdoor_siren_v6_run1'                             # 固定名（resume用）
ZIP_NAME   = f'dataset_{DATASET}.zip'

import os
for d in [DRIVE_DATA, DRIVE_LOGS, DRIVE_CKPT]:
    os.makedirs(d, exist_ok=True)
ZIP_PATH = f'{DRIVE_DATA}/{ZIP_NAME}'
assert os.path.exists(ZIP_PATH), f'⚠️ {ZIP_PATH} がありません。zip をアップロードしてください。'
print(f'OK: {ZIP_PATH} ({os.path.getsize(ZIP_PATH)/1e6:.0f} MB)')

## 3. リポジトリ clone

In [ ]:
import os

REPO = '/content/PSELDNets'

if not os.path.exists(f'{REPO}/src'):
    !git clone https://github.com/Jinbo-Hu/PSELDNets {REPO}
else:
    print(f'既にあります: {REPO}')

os.chdir(REPO)
print(f'CWD: {os.getcwd()}')

## 4. パッケージインストール

`numpy` / `h5py` / `scipy` / `torch` は Colab に最初から入っています。
ここでは**触らず**、不足しているものだけ追加します（v1-v4 と同一・再起動不要）。

In [ ]:
!pip install -q \
    librosa \
    soundfile \
    lightning==2.2.1 \
    hydra-core==1.3.2 \
    hydra-colorlog==1.2.0 \
    hydra-joblib-launcher==1.2.0 \
    torchmetrics==1.3.1

import numpy, lightning, torchmetrics, librosa
print(f'numpy {numpy.__version__} / lightning {lightning.__version__} / '
      f'torchmetrics {torchmetrics.__version__} / librosa {librosa.__version__}')

## 5. 事前学習チェックポイント（Drive キャッシュ → なければ HF から）

In [ ]:
import shutil

os.makedirs('ckpts', exist_ok=True)
CKPT = 'ckpts/mACCDOA-HTSAT-0.567.ckpt'
CACHE = f'{DRIVE_CKPT}/mACCDOA-HTSAT-0.567.ckpt'

if not os.path.exists(CKPT):
    if os.path.exists(CACHE):
        print('Drive キャッシュからコピー...')
        shutil.copy(CACHE, CKPT)
    else:
        print('HuggingFace からダウンロード...')
        from huggingface_hub import hf_hub_download
        src = hf_hub_download(repo_id='Jinbo-HU/PSELDNets',
                              filename='model/mACCDOA-HTSAT-0.567.ckpt',
                              repo_type='dataset')
        shutil.copy(src, CKPT)
        shutil.copy(CKPT, CACHE)   # 次回用に Drive へキャッシュ
print(f'OK: {CKPT} ({os.path.getsize(CKPT)/1e6:.0f} MB)')

## 6. データセット展開

zip は `datasets/...` 構成なのでリポジトリ直下で解凍するだけ。
クラス辞書 `cls_indices_train.tsv`（**本データセット専用の4クラス**: Siren/Horn/
BackupBeep/BikeBell）も同梱。

In [ ]:
import zipfile

if not os.path.exists(f'datasets/{DATASET}/foa'):
    with zipfile.ZipFile(ZIP_PATH) as z:
        z.extractall('.')
    print('解凍完了')

n_foa = len(os.listdir(f'datasets/{DATASET}/foa'))
n_meta = len(os.listdir(f'datasets/{DATASET}/metadata'))
n_cls = len(open('datasets/cls_indices_train.tsv').readlines())
print(f'foa: {n_foa} / metadata: {n_meta} / classes: {n_cls}')
assert n_foa == 320 and n_meta == 320 and n_cls == 4, '⚠️ ファイル数が想定と違います'

## 7. 設定ファイル2つを作成（新規追加のみ・リポジトリ既存ファイルは無編集）

In [ ]:
data_yaml = """audio_type: foa
audio_feature: logmelIV
sample_rate: 24000
nfft: 1024
n_mels: 64
hoplen: 240
window: hann

train_chunklen_sec: 10
train_hoplen_sec: 10
test_chunklen_sec: 10
test_hoplen_sec: 10

train_dataset:
  outdoor_siren_v6: [fold1_room1]
valid_dataset:
  outdoor_siren_v6: [fold2_room1]
test_dataset:
  outdoor_siren_v6: [fold2_room1]
"""

exp_yaml = """# @package _global_
defaults:
 - override /data: outdoor_siren_v6.yaml
 - override /loss: multi_accdoa.yaml
 - _self_

task_name: outdoor_siren_v6

model:
  batch_size: 8
  kwargs:
    pretrained_path: ckpts/mACCDOA-HTSAT-0.567.ckpt
    audioset_pretrain: false
  optimizer:
    kwargs: {lr: 0.0003}
  lr_scheduler:
    kwargs: {step_size: 60}

trainer:
  max_epochs: 100
  check_val_every_n_epoch: 5
"""

open('configs/data/outdoor_siren_v6.yaml', 'w').write(data_yaml)
open('configs/experiment/outdoor_siren_v6.yaml', 'w').write(exp_yaml)
print('wrote configs/data/outdoor_siren_v6.yaml')
print('wrote configs/experiment/outdoor_siren_v6.yaml')

## 8. 前処理（ラベル → HDF5、クリップ索引の作成。1分未満）

In [ ]:
IDX = f'_hdf5/data/24000fs/wav/dev/{DATASET}_10sChunklen_10sHoplen_train.csv'
if not os.path.exists(IDX):
    !python src/preproc.py dataset={DATASET}
else:
    print('既に前処理済み')
!head -3 {IDX}

## 9. 実行前チェック

In [ ]:
checks = [
    ('ckpts/mACCDOA-HTSAT-0.567.ckpt',     'チェックポイント'),
    ('datasets/cls_indices_train.tsv',      'クラス辞書 TSV (4クラス)'),
    (f'datasets/{DATASET}/foa',             'FOA データ (320)'),
    (f'datasets/{DATASET}/metadata',        'ラベル CSV (320)'),
    (f'configs/experiment/{DATASET}.yaml',  '実験設定'),
    (IDX,                                   'クリップ索引'),
]
for path, name in checks:
    ok = os.path.exists(path) and (not os.path.isdir(path) or len(os.listdir(path)) > 0)
    print(f'  [{"OK" if ok else "NG"}] {name}')

## 10. 学習（T4 で 60〜90 分見込み、100epoch）

- ログと ckpt は Drive（`PSELDNets_logs`）に直接書くので、セッションが切れても消えない
- `last.ckpt` があれば自動で続きから再開（固定 experiment_name 方式）
- **データを差し替えて再学習する場合は、上のcell4で`EXP_NAME`を新しい名前
  （`outdoor_siren_v6_run2`等）に変えてから実行すること**（v3のrun1 NaN崩壊事故を参照）
- **epoch数100の理由**: v5 run2（train60本・200epoch）はepoch114がベストだった。
  v6はtrain240本で1epochあたりの学習ステップが4倍になるため、同等の総ステップ数には
  約30epochで到達する。余裕を見て100epoch（v5 run2の約3倍の総ステップ）に設定
- メモリ不足になったら `model.batch_size=4` に下げて再実行

In [ ]:
LAST = f'{DRIVE_LOGS}/{DATASET}/runs/{EXP_NAME}/checkpoints/last.ckpt'
resume = f'ckpt_path={LAST}' if os.path.exists(LAST) else ''
print('resume:', resume or '(new run)')

!python src/train.py experiment={DATASET} \
    experiment_name={EXP_NAME} \
    paths.log_dir={DRIVE_LOGS} \
    {resume}

## 11. 結果の確認

`val/macro` の ER / F / LE / LR / **SELD_scr** を全エポック分抽出する。
**この val は学習に見せていない80本**（同一生成器の未見シーン、4クラス全体の集計値）。

**v6の見どころ（比較基準はv5 run2: ER 0.260 / F 77.3% / LE 14.1° / SELD_scr 0.152）**:
- v3水準（ER 0.042 / SELD_scr 0.027）まで改善 → v5の劣化は**クラス毎データ不足**が主因
- LEが14°前後のまま → 疎発音クラスのDOA問題は**データ量では解決しない本質的限界**
- 全体指標だけでは判定できないので、学習後に infer(mode=test) を実行し、ローカルの
  `scripts/step8_error_anatomy_mc.py --pred out/predictions_v6_run1 --ds out/dataset_outdoor_siren_v6`
  でクラス別の dir_err / miss / substitution を v5 と横並び比較する（手順はPROGRESS.md参照）

In [ ]:
import re

log_path = f'{DRIVE_LOGS}/{DATASET}/runs/{EXP_NAME}/train.log'
lines = [l for l in open(log_path, errors='ignore')
         if 'val/macro' in l or 'train: loss_all' in l]
print(f'--- {log_path} ---')
for l in lines:
    print(re.sub(r'\x1b\[[0-9;]*m', '', l).rstrip())

vals = [l for l in lines if 'val/macro' in l]
if vals:
    print('\n=== 最終 val/macro ===')
    print(re.sub(r'\x1b\[[0-9;]*m', '', vals[-1]).strip())

---
## メモ

- データ生成条件の全記録はローカルの `outdoor_seld_e2e/out/dataset_outdoor_siren_v6/`
  （`inspection.csv`=検品結果, `work/*/scene.json`=各クリップの条件、`hazard_class`欄に
  クラス名記録）
- v5との違いは**件数のみ**（80本→320本、各クラス20→80本）。構成・レンジ・音源・
  クラス辞書はすべて同一（`--v6`フラグ、scripts/step6_batch_scenes.py）
- 4クラスは各80本ずつ均等割当（idx%4）。妨害音・対象音とも全てクリーン合成（実録音は不使用）
- SNR/SIR はサイレン等の発音区間の W チャンネル基準で定義
- 結果を報告するときは ER / F / LE / LR / SELD_scr を併記する
- **experiment_nameの使い回しに注意**（v3のrun1 NaN崩壊事故参照。データを変えたら
  必ず新しいexperiment_nameにする）
- 学習後の誤り解剖: infer(mode=test) → Drive submissions/ の予測CSVをローカルに取得 →
  `scripts/step8_error_anatomy_mc.py`（v5 run2ですでに検証済みのマルチクラス対応版）
- **⚠️ v5とのデータ関係（評価時の注意）**: 生成シードの設計上、v5の全80本（val含む）は
  v6のtrain(mix001-080)にビット単位で含まれる（ハッシュ照合で確認済み）。したがって
  **v6モデルをv5のvalで評価するのは絶対に不可**（学習済みデータのため）。
  逆にv6のval（fold2の80本、idx240-319）は**v5モデルも一度も見ていない**ので、
  v5 run2のckptとv6のckptを同一のval 80本で対等比較（ペア比較）できる